In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
PROJECT_ROOT = Path(r"C:\Users\maria\Desktop\EEG-special-course")
PROCESSED_ROOT = PROJECT_ROOT / "data_processed"

features_file = PROCESSED_ROOT / "features_pre_ses1_with_posterior.csv"

In [3]:
features = pd.read_csv(features_file)

display(features.head())
print(features.shape)
print(features.columns.tolist())

,participant_id,age,condition,n_occipital_channels,n_posterior_channels,posterior_channels,n_epochs_kept,alpha_power_8_12,posterior_alpha_sum_8_12,fooof_alpha_cf,fooof_alpha_pw,fooof_alpha_bw,fooof_r2,fooof_error
0,sub-001,60,EyesClosed,3,19,"P7,P3,Pz,P4,P8,PO9,O1,Oz,O2,PO10,P5,P1,P2,P6,P...",77,-10.257854,-9.067168,10.109051,1.711548,1.850375,0.958282,0.085840
1,sub-001,60,EyesOpen,3,19,"P7,P3,Pz,P4,P8,PO9,O1,Oz,O2,PO10,P5,P1,P2,P6,P...",35,-11.266725,-9.949122,10.994871,0.382865,3.216583,0.973884,0.050167
2,sub-002,67,EyesClosed,3,19,"P7,P3,Pz,P4,P8,PO9,O1,Oz,O2,PO10,P5,P1,P2,P6,P...",81,-10.221930,-9.088553,9.204659,1.745596,2.435050,0.966918,0.075781
3,sub-002,67,EyesOpen,3,19,"P7,P3,Pz,P4,P8,PO9,O1,Oz,O2,PO10,P5,P1,P2,P6,P...",47,-11.493045,-10.101173,8.467325,0.338656,1.327376,0.977051,0.043199
4,sub-003,44,EyesClosed,3,19,"P7,P3,Pz,P4,P8,PO9,O1,Oz,O2,PO10,P5,P1,P2,P6,P...",107,-10.989451,-9.893156,11.352306,1.386752,1.756418,0.967573,0.067330


(1216, 14)
['participant_id', 'age', 'condition', 'n_occipital_channels', 'n_posterior_channels', 'posterior_channels', 'n_epochs_kept', 'alpha_power_8_12', 'posterior_alpha_sum_8_12', 'fooof_alpha_cf', 'fooof_alpha_pw', 'fooof_alpha_bw', 'fooof_r2', 'fooof_error']


In [4]:
features.groupby("condition")[
    ["alpha_power_8_12", "posterior_alpha_sum_8_12"]
].describe()

alpha_power_8_12                                             \
                      count       mean       std        min        25%   
condition                                                                
EyesClosed            604.0 -10.733910  0.592002 -12.100806 -11.167402   
EyesOpen              591.0 -11.351595  0.402918 -12.143214 -11.649687   

                                           posterior_alpha_sum_8_12  \
                  50%       75%        max                    count   
condition                                                             
EyesClosed -10.677601 -10.27047  -9.480998                    604.0   
EyesOpen   -11.420396 -11.11035 -10.218024                    591.0   

                                                                            \
                 mean       std        min        25%        50%       75%   
condition                                                                    
EyesClosed  -9.509320  0.565099 -10.887405  -9.932720  -9.459034 -9.061417   
EyesOpen   -10.084499  0.413192 -10.935661 -10.402913 -10.145948 -9.817239   

                      
                 max  
condition             
EyesClosed -8.344808  
EyesOpen   -8.851775

In [5]:
posterior_wide = (
    features
    .pivot_table(
        index="participant_id",
        columns="condition",
        values="alpha_power_8_12"
    )
    .reset_index()
)

posterior_wide = posterior_wide.dropna(subset=["EyesClosed", "EyesOpen"]).copy()

posterior_wide.head()

condition,participant_id,EyesClosed,EyesOpen
0,sub-001,-10.257854,-11.266725
1,sub-002,-10.221930,-11.493045
2,sub-003,-10.989451,-11.924012
3,sub-004,-9.980468,-11.574404
4,sub-005,-10.728284,-11.493758


In [6]:
posterior_wide["posterior_alpha_diff_ec_minus_eo"] = (
    posterior_wide["EyesClosed"] - posterior_wide["EyesOpen"]
)

posterior_wide["predicted_closed_condition"] = np.where(
    posterior_wide["EyesClosed"] > posterior_wide["EyesOpen"],
    "EyesClosed",
    "EyesOpen"
)

posterior_wide["correct"] = (
    posterior_wide["predicted_closed_condition"] == "EyesClosed"
)

display(posterior_wide.head())

condition,participant_id,EyesClosed,EyesOpen,posterior_alpha_diff_ec_minus_eo,predicted_closed_condition,correct
0,sub-001,-10.257854,-11.266725,1.008871,EyesClosed,True
1,sub-002,-10.221930,-11.493045,1.271115,EyesClosed,True
2,sub-003,-10.989451,-11.924012,0.934561,EyesClosed,True
3,sub-004,-9.980468,-11.574404,1.593936,EyesClosed,True
4,sub-005,-10.728284,-11.493758,0.765474,EyesClosed,True


In [7]:
accuracy = posterior_wide["correct"].mean()

n_total = len(posterior_wide)
n_correct = posterior_wide["correct"].sum()
n_incorrect = n_total - n_correct

print(f"Simple posterior-alpha baseline accuracy: {accuracy:.3f}")
print(f"Correct: {n_correct}/{n_total}")
print(f"Incorrect: {n_incorrect}/{n_total}")

Simple posterior-alpha baseline accuracy: 0.959
Correct: 566/590
Incorrect: 24/590


In [8]:
incorrect = posterior_wide[posterior_wide["correct"] == False].copy()

print(f"Number of incorrect classifications: {len(incorrect)}")

display(
    incorrect
    .sort_values("posterior_alpha_diff_ec_minus_eo")
    .head(20)
)

Number of incorrect classifications: 24


condition,participant_id,EyesClosed,EyesOpen,posterior_alpha_diff_ec_minus_eo,predicted_closed_condition,correct
66,sub-069,-11.952282,-11.691369,-0.260912,EyesOpen,False
475,sub-479,-11.733419,-11.500425,-0.232994,EyesOpen,False
427,sub-430,-11.926298,-11.725005,-0.201293,EyesOpen,False
365,sub-368,-11.876304,-11.729801,-0.146502,EyesOpen,False
396,sub-399,-11.776867,-11.630542,-0.146326,EyesOpen,False
240,sub-243,-11.148970,-11.015928,-0.133041,EyesOpen,False
502,sub-506,-11.834390,-11.702375,-0.132014,EyesOpen,False
17,sub-020,-11.751119,-11.634742,-0.116377,EyesOpen,False
412,sub-415,-11.978743,-11.871564,-0.107179,EyesOpen,False
8,sub-009,-10.612547,-10.506169,-0.106378,EyesOpen,False


In [9]:
correct = posterior_wide[posterior_wide["correct"] == True].copy()

display(
    correct
    .sort_values("posterior_alpha_diff_ec_minus_eo", ascending=False)
    .head(20)
)

condition,participant_id,EyesClosed,EyesOpen,posterior_alpha_diff_ec_minus_eo,predicted_closed_condition,correct
99,sub-102,-10.036988,-11.795830,1.758842,EyesClosed,True
460,sub-464,-10.092355,-11.833347,1.740992,EyesClosed,True
356,sub-359,-9.832980,-11.565186,1.732206,EyesClosed,True
370,sub-373,-9.865343,-11.533582,1.668239,EyesClosed,True
395,sub-398,-10.034218,-11.658828,1.624610,EyesClosed,True
3,sub-004,-9.980468,-11.574404,1.593936,EyesClosed,True
237,sub-240,-9.892149,-11.428124,1.535975,EyesClosed,True
531,sub-535,-9.795406,-11.323697,1.528292,EyesClosed,True
508,sub-512,-9.943344,-11.454999,1.511655,EyesClosed,True
143,sub-146,-10.037375,-11.511426,1.474051,EyesClosed,True


In [10]:
occipital_wide = (
    features
    .pivot_table(
        index="participant_id",
        columns="condition",
        values="alpha_power_8_12"
    )
    .reset_index()
)

occipital_wide = occipital_wide.dropna(subset=["EyesClosed", "EyesOpen"]).copy()

occipital_wide["alpha_diff_ec_minus_eo"] = (
    occipital_wide["EyesClosed"] - occipital_wide["EyesOpen"]
)

occipital_wide["predicted_closed_condition"] = np.where(
    occipital_wide["EyesClosed"] > occipital_wide["EyesOpen"],
    "EyesClosed",
    "EyesOpen"
)

occipital_wide["correct"] = (
    occipital_wide["predicted_closed_condition"] == "EyesClosed"
)

occipital_accuracy = occipital_wide["correct"].mean()

print(f"Occipital alpha paired baseline accuracy: {occipital_accuracy:.3f}")
print(f"Correct: {occipital_wide['correct'].sum()}/{len(occipital_wide)}")
print(f"Incorrect: {(~occipital_wide['correct']).sum()}/{len(occipital_wide)}")

display(occipital_wide.head())

Occipital alpha paired baseline accuracy: 0.959
Correct: 566/590
Incorrect: 24/590


condition,participant_id,EyesClosed,EyesOpen,alpha_diff_ec_minus_eo,predicted_closed_condition,correct
0,sub-001,-10.257854,-11.266725,1.008871,EyesClosed,True
1,sub-002,-10.221930,-11.493045,1.271115,EyesClosed,True
2,sub-003,-10.989451,-11.924012,0.934561,EyesClosed,True
3,sub-004,-9.980468,-11.574404,1.593936,EyesClosed,True
4,sub-005,-10.728284,-11.493758,0.765474,EyesClosed,True


In [12]:
posterior_wide = (
    features
    .pivot_table(
        index="participant_id",
        columns="condition",
        values="posterior_alpha_sum_8_12"
    )
    .reset_index()
)

posterior_wide = posterior_wide.dropna(subset=["EyesClosed", "EyesOpen"]).copy()

posterior_wide["posterior_alpha_diff_ec_minus_eo"] = (
    posterior_wide["EyesClosed"] - posterior_wide["EyesOpen"]
)

posterior_wide["predicted_closed_condition"] = np.where(
    posterior_wide["EyesClosed"] > posterior_wide["EyesOpen"],
    "EyesClosed",
    "EyesOpen"
)

posterior_wide["correct"] = (
    posterior_wide["predicted_closed_condition"] == "EyesClosed"
)

posterior_accuracy = posterior_wide["correct"].mean()

print(f"Posterior alpha paired baseline accuracy: {posterior_accuracy:.3f}")
print(f"Correct: {posterior_wide['correct'].sum()}/{len(posterior_wide)}")
print(f"Incorrect: {(~posterior_wide['correct']).sum()}/{len(posterior_wide)}")

display(posterior_wide.head())

Posterior alpha paired baseline accuracy: 0.958
Correct: 565/590
Incorrect: 25/590


condition,participant_id,EyesClosed,EyesOpen,posterior_alpha_diff_ec_minus_eo,predicted_closed_condition,correct
0,sub-001,-9.067168,-9.949122,0.881954,EyesClosed,True
1,sub-002,-9.088553,-10.101173,1.012621,EyesClosed,True
2,sub-003,-9.893156,-10.685098,0.791942,EyesClosed,True
3,sub-004,-8.827828,-10.386710,1.558882,EyesClosed,True
4,sub-005,-9.338288,-10.225608,0.887320,EyesClosed,True


In [13]:
baseline_comparison = pd.DataFrame({
    "baseline": [
        "Occipital alpha paired rule",
        "Posterior alpha paired rule"
    ],
    "feature": [
        "alpha_power_8_12",
        "posterior_alpha_sum_8_12"
    ],
    "accuracy": [
        occipital_accuracy,
        posterior_accuracy
    ],
    "n_participants": [
        len(occipital_wide),
        len(posterior_wide)
    ],
    "n_correct": [
        occipital_wide["correct"].sum(),
        posterior_wide["correct"].sum()
    ],
    "n_incorrect": [
        (~occipital_wide["correct"]).sum(),
        (~posterior_wide["correct"]).sum()
    ]
})

display(baseline_comparison)

,baseline,feature,accuracy,n_participants,n_correct,n_incorrect
0,Occipital alpha paired rule,alpha_power_8_12,0.959322,590,566,24
1,Posterior alpha paired rule,posterior_alpha_sum_8_12,0.957627,590,565,25
